In [1]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aliu917 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Colab

In [2]:
import os
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [3]:
DRIVE_PATH = '/content/gdrive/MyDrive/cs329h/code'
DRIVE_PYTHON_PATH = DRIVE_PATH.replace('\\', '')
if not os.path.exists(DRIVE_PYTHON_PATH):
  %mkdir $DRIVE_PATH

## the space in `My Drive` causes some issues,
## make a symlink to avoid this
SYM_PATH = '/content/cs329h'
if not os.path.exists(SYM_PATH):
  !ln -s $DRIVE_PATH $SYM_PATH

In [4]:
%cd cs329h

/content/gdrive/MyDrive/cs329h/code


In [5]:
import sys
sys.path.append("/content/gdrive/MyDrive/cs329h/code")

In [6]:
!pip install -U bitsandbytes
%pip install -r requirements_colab.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.4/402.4 kB 37.8 MB/s eta 0:00:00
  Attempting uninstall: sentry-sdk
    Found existing installation: sentry-sdk 2.45.0
    Uninstalling sentry-sdk-2.45.0:
      Successfully uninstalled sentry-sdk-2.45.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.11.0
    Uninstalling accelerate-1.11.0:
      Successfully uninstalled accelerate-1.11.0
  Attempting uninstall: peft
    Found existing installation: peft 0.18.0
    Uninstalling peft-0.18.0:
      Successfully uninstalled peft-0.18.0


In [7]:
import argparse
import sys

from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import torch
import wandb
import os
import numpy as np

from custom_dpo import dpo_step
from dpo_dataset import DpoJsonlDataset
from tqdm import tqdm

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [8]:
def dpo_collate(batch, tokenizer, add_generation_prompt=False):
    chosen_texts = [
        [{"role": "user", "content": item["prompt"]},
         {"role": "assistant", "content": item["chosen_text"]}]
        for item in batch
    ]
    rejected_texts = [
        [{"role": "user", "content": item["prompt"]},
         {"role": "assistant", "content": item["rejected_text"]}]
        for item in batch
    ]
    sample_fracs = [item["sample_frac"] for item in batch]

    chosen_encoding_ids = tokenizer.apply_chat_template(
        chosen_texts, tokenize=True, add_generation_prompt=add_generation_prompt
    )
    rejected_encoding_ids = tokenizer.apply_chat_template(
        rejected_texts, tokenize=True, add_generation_prompt=add_generation_prompt
    )
    chosen_encodings = [{"input_ids": ids} for ids in chosen_encoding_ids]
    rejected_encodings = [{"input_ids": ids} for ids in rejected_encoding_ids]

    chosen_batch = tokenizer.pad(chosen_encodings, padding=True, return_tensors="pt")
    rejected_batch = tokenizer.pad(rejected_encodings, padding=True, return_tensors="pt")
    # metadata = [item["metadata"] for item in batch]

    return {
        "chosen": chosen_batch,
        "rejected": rejected_batch,
        "sample_frac": torch.tensor(sample_fracs),
    }

In [9]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Enables 4-bit quantization
    bnb_4bit_use_double_quant=True,  # Use double quantization for potentially higher accuracy (optional)
    bnb_4bit_quant_type="nf4",  # Quantization type (specifics depend on hardware and library)
    bnb_4bit_compute_dtype=torch.bfloat16  # Compute dtype for improved efficiency (optional)
)

# RUN CONFIG

In [10]:
run_name = "run_sample_noips"

In [11]:
default_config = {
    "model_name": "google/gemma-2-2b-it",
    "device": device,
    "learning_rate": 2e-6,
    "num_epochs": 5,
    "beta": 1,
    "max_seq_len": 512,
    "batch_size": 1,
    "ips": False,
    "sample_data": True,
}

In [12]:
def seed():
    seed = 1

    np.random.seed(seed)
    torch.manual_seed(seed)

    if device == "cuda":
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

seed()

In [13]:
run = wandb.init(
    project="dpo-finetune",
    name=run_name,
    config=default_config
)
if wandb.config.sample_data:
    path = "data/sampled_dpo_dataset_200.jsonl"
    val_path = "data/sampled_dpo_dataset_val_100.jsonl"
else:
    path = "data/all_dpo_dataset_200.jsonl"
    val_path = "data/all_dpo_dataset_val_100.jsonl"


In [14]:
model_name = default_config["model_name"]
tokenizer = AutoTokenizer.from_pretrained(model_name, token=os.environ.get("HUGGINGFACE_TOKEN"))
ref_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=os.environ.get("HUGGINGFACE_TOKEN"),
    # low_cpu_mem_usage=True
    quantization_config=bnb_config,
)
ref_model.eval()

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear4bit(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear4bit(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear4bit(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedfor

# Training

In [15]:
train_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=os.environ.get("HUGGINGFACE_TOKEN"),
    # low_cpu_mem_usage=True,
    quantization_config=bnb_config,
)
lora_config = LoraConfig(
    r=64,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
)
train_model = get_peft_model(train_model, lora_config)
train_model.train()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): Gemma2ForCausalLM(
      (model): Gemma2Model(
        (embed_tokens): Embedding(256000, 2304, padding_idx=0)
        (layers): ModuleList(
          (0-25): 26 x Gemma2DecoderLayer(
            (self_attn): Gemma2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2304, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2304, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
           

In [16]:
dataset = DpoJsonlDataset(path, tokenizer=tokenizer, max_length=wandb.config.max_seq_len)
dataloader = DataLoader(dataset, batch_size=wandb.config.batch_size, shuffle=True, num_workers=0, collate_fn=lambda batch: dpo_collate(batch, tokenizer))
optimizer = torch.optim.Adam(train_model.parameters(), lr=wandb.config.learning_rate)

val_dataset = DpoJsonlDataset(val_path, tokenizer=tokenizer, max_length=wandb.config.max_seq_len)
val_dataloader = DataLoader(val_dataset, batch_size=wandb.config.batch_size, shuffle=True, num_workers=0, collate_fn=lambda batch: dpo_collate(batch, tokenizer))

orig_val_dataset = DpoJsonlDataset("data/all_dpo_dataset_val_100.jsonl", tokenizer=tokenizer, max_length=wandb.config.max_seq_len)
orig_val_dataloader = DataLoader(orig_val_dataset, batch_size=wandb.config.batch_size, shuffle=True, num_workers=0, collate_fn=lambda batch: dpo_collate(batch, tokenizer))


In [17]:
def run_validation(model):
  total_loss = 0
  for step, data in enumerate(tqdm(val_dataloader, total=len(val_dataloader), desc=f"val")):
      # if step > 5:
      #   break
      preferred_chat_ids = data["chosen"]["input_ids"].to(train_model.device)
      preferred_mask = data["chosen"]["attention_mask"].to(train_model.device)
      nonpreferred_chat_ids = data["rejected"]["input_ids"].to(train_model.device)
      nonpreferred_mask = data["rejected"]["attention_mask"].to(train_model.device)

      ips_weight = None
      if wandb.config.ips:
          ips_weight = 1 / data["sample_frac"].to(train_model.device)

      with torch.no_grad():
          loss = dpo_step(train_model, ref_model, preferred_chat_ids, nonpreferred_chat_ids, preferred_mask, nonpreferred_mask, wandb.config.beta, ips_weight)
          total_loss += loss.item()

  print("val_loss:", total_loss / len(val_dataloader))
  return total_loss / len(val_dataloader)

def run_orig_validation(model):
  total_loss = 0
  for step, data in enumerate(tqdm(orig_val_dataloader, total=len(orig_val_dataloader), desc=f"val")):
      # if step > 5:
      #   break
      preferred_chat_ids = data["chosen"]["input_ids"].to(train_model.device)
      preferred_mask = data["chosen"]["attention_mask"].to(train_model.device)
      nonpreferred_chat_ids = data["rejected"]["input_ids"].to(train_model.device)
      nonpreferred_mask = data["rejected"]["attention_mask"].to(train_model.device)

      ips_weight = None
      with torch.no_grad():
          loss = dpo_step(train_model, ref_model, preferred_chat_ids, nonpreferred_chat_ids, preferred_mask, nonpreferred_mask, wandb.config.beta, ips_weight)
          total_loss += loss.item()

  print("val_loss:", total_loss / len(orig_val_dataloader))
  return total_loss / len(orig_val_dataloader)

In [ ]:
train_model.train()
ref_model.eval()

val_loss = run_validation(train_model)
orig_val_loss = val_loss
if wandb.config.sample_data:
    orig_val_loss = run_orig_validation(train_model)
wandb.log({
    "orig_val_loss" : orig_val_loss,
    "val_loss": orig_val_loss,
    "ipw_val_loss": val_loss,
    "epoch": 0
})

for epoch in range(wandb.config.num_epochs):

    total_loss = 0
    for step, data in enumerate(tqdm(dataloader, total=len(dataloader), desc=f"Epoch {epoch+1}")):
        # if step > 5:
        #   break
        preferred_chat_ids = data["chosen"]["input_ids"].to(train_model.device)
        preferred_mask = data["chosen"]["attention_mask"].to(train_model.device)
        nonpreferred_chat_ids = data["rejected"]["input_ids"].to(train_model.device)
        nonpreferred_mask = data["rejected"]["attention_mask"].to(train_model.device)

        ips_weight = None
        if wandb.config.ips:
            ips_weight = 1 / data["sample_frac"].to(train_model.device)

        loss = dpo_step(train_model, ref_model, preferred_chat_ids, nonpreferred_chat_ids, preferred_mask, nonpreferred_mask, wandb.config.beta, ips_weight)
        # if ips_weight > 1:
        #     print("ips weight:", ips_weight)
        #     print(loss)
        #     with torch.no_grad():
        #       print(dpo_step(train_model, ref_model, preferred_chat_ids, nonpreferred_chat_ids, preferred_mask, nonpreferred_mask, wandb.config.beta, None))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        if len(data) > 1:
            # print(f"Step: {step+1}: {loss.item():.4f}")
            wandb.log({
                "loss": loss.item(),
                "step": step
            })

    print("epoch train loss:", total_loss / len(dataloader))
    val_loss = run_validation(train_model)
    orig_val_loss = val_loss
    if wandb.config.sample_data:
        orig_val_loss = run_orig_validation(train_model)
    wandb.log({
        "orig_val_loss" : orig_val_loss,
        "val_loss": orig_val_loss,
        "ipw_val_loss": val_loss,
        "train_loss": total_loss / len(dataloader),
        "epoch": epoch+1
    })

    if True:
        print("saving model")
        LOCAL_SAVE_PATH = f"./dpo_merged_checkpoint_e{epoch}"
        ADAPTER_PATH = f"./dpo_adapter_checkpoint_e{epoch}"
        train_model.save_pretrained(ADAPTER_PATH)
        artifact_name = "dpo-adapter-checkpoints-" + run_name
        artifact = wandb.Artifact(
            name=artifact_name,
            type="model",
            description=f"DPO LoRA Adapter checkpoint from Epoch {epoch}, Step {step+1}"
        )
        artifact.add_dir(ADAPTER_PATH)
        wandb.log_artifact(artifact, aliases=[f"e{epoch}"])
        print("done saving model")


val: 100%|██████████| 100/100 [00:46<00:00,  2.14it/s]


val_loss: 0.69140625


val: 100%|██████████| 100/100 [00:46<00:00,  2.14it/s]


val_loss: 0.69140625


Epoch 1: 100%|██████████| 3211/3211 [38:27<00:00,  1.39it/s]


...

val_loss: 2.580366239948953


val: 100%|██████████| 100/100 [00:47<00:00,  2.12it/s]


val_loss: 2.149445581082255
saving model


wandb: Adding directory to artifact (dpo_adapter_checkpoint_e4)... Done. 0.2s


done saving model


In [ ]:
run_orig_validation(train_model)

val: 100%|██████████| 100/100 [00:47<00:00,  2.12it/s]

val_loss: 1.6785823001898825


1.6785823001898825

In [ ]:
from collections import Counter

def run_eval_distr(model, dataloader):
  seed()
  total_loss_sum = Counter()
  total_loss_count = Counter()
  for step, data in enumerate(tqdm(dataloader, total=len(dataloader), desc=f"val")):
      preferred_chat_ids = data["chosen"]["input_ids"].to(train_model.device)
      preferred_mask = data["chosen"]["attention_mask"].to(train_model.device)
      nonpreferred_chat_ids = data["rejected"]["input_ids"].to(train_model.device)
      nonpreferred_mask = data["rejected"]["attention_mask"].to(train_model.device)

      category = round(data["sample_frac"].item(), 1)
      with torch.no_grad():
          loss = dpo_step(train_model, ref_model, preferred_chat_ids, nonpreferred_chat_ids, preferred_mask, nonpreferred_mask, wandb.config.beta, None)
          total_loss_sum[category] += loss.item()
          total_loss_count[category] += 1
  final_distr = {k: v / total_loss_count[k] for k, v in total_loss_sum.items()}
  return final_distr

In [ ]:
run_eval_distr(train_model, val_dataloader)